In [ ]:
!pip -q install fastapi uvicorn nest-asyncio transformers accelerate sentencepiece


In [ ]:
import os
import re
import time
import threading
import subprocess
import socket

import nest_asyncio
import torch
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
import traceback

from google.colab import userdata

nest_asyncio.apply()

HF_TOKEN = userdata.get("HF_TOKEN")

def pick_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

PORT = int(os.getenv("API_PORT", "0")) or pick_free_port()
print(f"Using API port: {PORT}")

QWEN_MODEL_ID = os.getenv("QWEN_MODEL_ID", "Qwen/Qwen3-4B")
TRANSLATE_MODEL_ID = os.getenv("TRANSLATE_MODEL_ID", "facebook/mbart-large-50-many-to-many-mmt")

INTENT_SYSTEM = (
    "You are an intent classifier for a hotel chat system. "
    "Return exactly one label: booking, faq, complex. Output only the label."
)

REWRITE_SYSTEM = (
    "Rewrite the following Burmese text to be polite, friendly, and professional. "
    "Return only the rewritten Burmese text in Myanmar script. "
    "Do not include English or explanations."
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

MBART_LANG_MAP = {
    "en": "en_XX",
    "my": "my_MM",
}

app = FastAPI()

qwen_tokenizer = None
qwen_model = None
enmy_tokenizer = None
enmy_model = None
myen_tokenizer = None
myen_model = None


Using API port: 40325


In [ ]:
def load_models():
    global qwen_tokenizer, qwen_model, enmy_tokenizer, enmy_model, myen_tokenizer, myen_model

    print("Loading models...")

    # Load Qwen
    qwen_tokenizer = AutoTokenizer.from_pretrained(
        QWEN_MODEL_ID, token=HF_TOKEN, trust_remote_code=True
    )
    qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_ID,
        dtype=DTYPE,
        device_map="auto",
        token=HF_TOKEN,
        trust_remote_code=True
    )
    qwen_model.eval()
    print("Qwen loaded")

    # Load one shared mBART model for both EN <-> MY translation
    shared_tokenizer = AutoTokenizer.from_pretrained(
        TRANSLATE_MODEL_ID, token=HF_TOKEN
    )
    shared_model = AutoModelForSeq2SeqLM.from_pretrained(
        TRANSLATE_MODEL_ID,
        dtype=DTYPE,
        device_map="auto",
        token=HF_TOKEN
    )
    shared_model.eval()

    enmy_tokenizer = shared_tokenizer
    enmy_model = shared_model
    myen_tokenizer = shared_tokenizer
    myen_model = shared_model
    print("Shared translation model loaded")


In [ ]:
load_models()


Loading models...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Shared translation model loaded


In [ ]:
class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 512
    temperature: float = 0.2


class TranslateRequest(BaseModel):
    text: str
    source_lang: str = "mya_Mymr"
    target_lang: str = "eng_Latn"


class IntentRequest(BaseModel):
    text: str


class RewriteRequest(BaseModel):
    text: str


def chat_generate(system_text, user_text, max_tokens=512, temperature=0.2):
    if qwen_model is None or qwen_tokenizer is None:
        raise RuntimeError("Qwen model not loaded yet")

    no_think = "Answer directly. Do not output analysis, thinking tags, or <think>."

    messages = []
    if system_text:
        messages.append({"role": "system", "content": system_text})
    messages.append({"role": "system", "content": no_think})
    messages.append({"role": "user", "content": user_text})

    encoded = qwen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    )

    input_ids = None
    attention_mask = None

    if isinstance(encoded, torch.Tensor):
        input_ids = encoded
    else:
        if hasattr(encoded, "get"):
            input_ids = encoded.get("input_ids")
            attention_mask = encoded.get("attention_mask")
        if input_ids is None and hasattr(encoded, "input_ids"):
            input_ids = encoded.input_ids
        if attention_mask is None and hasattr(encoded, "attention_mask"):
            attention_mask = encoded.attention_mask

    if input_ids is None:
        raise RuntimeError("Tokenizer did not return input_ids")

    if not isinstance(input_ids, torch.Tensor):
        input_ids = torch.as_tensor(input_ids)
    input_ids = input_ids.to(qwen_model.device)

    if attention_mask is not None:
        if not isinstance(attention_mask, torch.Tensor):
            attention_mask = torch.as_tensor(attention_mask)
        attention_mask = attention_mask.to(qwen_model.device)

    with torch.inference_mode():
        outputs = qwen_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=temperature > 0
        )

    new_tokens = outputs[0][input_ids.shape[-1]:]
    text = qwen_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    # Strip any chain-of-thought output (handles both "<think>...</think>" and "<think>..." cases)
    if "<think>" in text:
        # remove everything up to the end tag if present
        if "</think>" in text:
            text = text.split("</think>", 1)[1].strip()
        else:
            # no end tag: drop the first line or everything before first newline
            text = text.split("\n", 1)[-1].strip()

    # If it still starts with "<think>", drop it hard
    while text.startswith("<think>"):
        text = text[len("<think>"):].strip()

    return text


BURMESE_CHAR_REGEX = re.compile(r"[\u1000-\u109F]")

def normalize_lang(code):
    if not code:
        return ""
    return str(code).strip().lower().replace("-", "_")

def is_burmese_lang(code):
    c = normalize_lang(code)
    return c.startswith("mya") or c.startswith("my") or c == "my_mm"

def is_english_lang(code):
    c = normalize_lang(code)
    return c.startswith("eng") or c == "en" or c.startswith("en_")

def contains_burmese(text):
    return bool(BURMESE_CHAR_REGEX.search(text or ""))

def extract_burmese_text(text):
    if not text:
        return ""
    raw = str(text)
    lines = [line.strip() for line in raw.splitlines() if line.strip()]
    burmese_lines = [line for line in lines if contains_burmese(line)]
    if burmese_lines:
        return " ".join(burmese_lines).strip()
    parts = re.split(r"(?<=[\u104b!?])\s+|\n+", raw)
    keep = [part.strip() for part in parts if contains_burmese(part)]
    return " ".join(keep).strip()

def prefer_burmese_output(text, fallback=""):
    cleaned = extract_burmese_text(text)
    if cleaned:
        return cleaned
    return fallback or ""

def resolve_mbart_lang(source_lang, target_lang, text=""):
    if is_english_lang(source_lang):
        src = MBART_LANG_MAP["en"]
    elif is_burmese_lang(source_lang):
        src = MBART_LANG_MAP["my"]
    elif contains_burmese(text):
        src = MBART_LANG_MAP["my"]
    else:
        src = MBART_LANG_MAP["en"]

    if is_english_lang(target_lang):
        tgt = MBART_LANG_MAP["en"]
    elif is_burmese_lang(target_lang):
        tgt = MBART_LANG_MAP["my"]
    elif src == MBART_LANG_MAP["en"]:
        tgt = MBART_LANG_MAP["my"]
    else:
        tgt = MBART_LANG_MAP["en"]

    return src, tgt

def translate_text(text, source_lang, target_lang, max_new_tokens=512):
    if not text:
        return ""

    if enmy_model is None or enmy_tokenizer is None:
        raise RuntimeError("Shared translation model not loaded yet")

    src_lang, tgt_lang = resolve_mbart_lang(source_lang, target_lang, text)

    if hasattr(enmy_tokenizer, "src_lang"):
        enmy_tokenizer.src_lang = src_lang

    inputs = enmy_tokenizer(text, return_tensors="pt", src_lang=src_lang)
    if hasattr(inputs, "to"):
        inputs = inputs.to(enmy_model.device)

    gen_kwargs = {"max_new_tokens": max_new_tokens}
    if hasattr(enmy_tokenizer, "lang_code_to_id"):
        lang_id = enmy_tokenizer.lang_code_to_id.get(tgt_lang)
        if lang_id is not None:
            gen_kwargs["forced_bos_token_id"] = lang_id

    with torch.inference_mode():
        outputs = enmy_model.generate(**inputs, **gen_kwargs)

    translated = enmy_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if tgt_lang == MBART_LANG_MAP["my"]:
        return prefer_burmese_output(translated, translated)

    return translated


In [ ]:
@app.get("/health")
def health():
    return {
        "status": "ok",
        "qwen_loaded": qwen_model is not None,
        "enmy_loaded": enmy_model is not None,
        "myen_loaded": myen_model is not None
    }


@app.post("/generate")
def generate(req: GenerateRequest):
    try:
        text = chat_generate(None, req.prompt, max_tokens=req.max_tokens, temperature=req.temperature)
        return {"text": text}
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": repr(e), "type": type(e).__name__})


@app.post("/intent")
def intent(req: IntentRequest):
    try:
        text = chat_generate(INTENT_SYSTEM, req.text, max_tokens=8, temperature=0.0)
        return {"intent": text}
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": repr(e), "type": type(e).__name__})


@app.post("/rewrite")
def rewrite(req: RewriteRequest):
    try:
        text = chat_generate(REWRITE_SYSTEM, req.text, max_tokens=384, temperature=0.2)
        text = prefer_burmese_output(text, req.text)
        return {"text": text}
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": repr(e), "type": type(e).__name__})


@app.post("/translate")
def translate(req: TranslateRequest):
    try:
        text = translate_text(req.text, req.source_lang, req.target_lang)
        return {"text": text}
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": repr(e), "type": type(e).__name__})


In [ ]:
server_thread = None

def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0

def run_api():
    import asyncio
    import uvicorn

    if is_port_open(PORT):
        print(f"API already running on port {PORT}")
        return

    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
    server = uvicorn.Server(config)
    server.install_signal_handlers = False

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())

global server_thread
if server_thread is None or not server_thread.is_alive():
    server_thread = threading.Thread(target=run_api, daemon=True)
    server_thread.start()
else:
    print("API thread already running")


In [ ]:
import requests

def wait_for_api():
    for _ in range(60):
        try:
            r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=1)
            if r.ok:
                print("Local API OK")
                return
        except Exception:
            pass
        time.sleep(0.5)
    raise RuntimeError(f"API not reachable on 127.0.0.1:{PORT}")

wait_for_api()


INFO:     Started server process [2788]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:40325 (Press CTRL+C to quit)


INFO:     127.0.0.1:42044 - "GET /health HTTP/1.1" 200 OK
Local API OK


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(80):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.25)
        continue
    match = re.search(r"https://[\w.-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("Public URL:", public_url)


Public URL: https://palestinian-photo-salem-supports.trycloudflare.com


In [ ]:
!curl -s -X POST http://127.0.0.1:{PORT}/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt":"hello","max_tokens":5,"temperature":0.2}'
